[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/05_coral.ipynb)

In [ ]:
# On Colab, install the CMGDB fork (a prebuilt wheel; not on PyPI)
# and the paper package. Running locally uses the project venv as is.
import sys

if "google.colab" in sys.modules:
    !pip install -q cmgdb==1.3.3+fork.2 --find-links https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3+fork.2
    !pip install -q git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.4 - Red Coral population model

## What this notebook shows

A demographic model of **Mediterranean red coral** (paper section 5.4) built
from real field data, with **thirteen age classes** (a 13-dimensional system).
We learn a **one-dimensional** latent model -- the bistability lives on a single
latent coordinate -- and compute its Morse graph, which has two minimal nodes
(two stable equilibria).

The default loads the model trained with **adaptive sampling**: the base design
$\mathcal{D}(500)$ augmented with 300 adaptively-chosen samples (section 5.4.2).
The histograms at the end compare the base design against the augmented one.

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes, run in
stages: model, training curves, Morse graph.

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | train a fresh model with your `OVERRIDES`, then compute its Morse graph at `SUBDIV` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

> **Retrain cost:** a full coral cell trains and computes Morse on the order of ~50-87 minutes.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # 1-D latent: SUBDIV/BOX_SCALE have little effect; this is a demographic model
OVERRIDES = {}             # MODE="retrain": config overrides (pydantic-validated), e.g.
                           #   {"training": {"epochs": 300}, "cmgdb": {"subdiv_max": 20}}
MORSE_OVERRIDES = {}       # extra cmgdb fields for the Morse cell, e.g.
                           #   {"compute_roa": True} (slow) or {"padding": False}
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
TRAIN_FILE = "train_500_300_adaptive"   # which sampling design to load
# ===========================================================================

## Model

Load the paper's trained model, or train a fresh one.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

REPLAY_CONFIG  = "coral_adaptive"
RETRAIN_CONFIG = "coral_basic"

if MODE in ("replay", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED, train_file=TRAIN_FILE)
elif MODE == "retrain":
    # Training only. CMGDB runs further down, so a long training run survives a
    # Morse computation that needs a different grid.
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        train_file=TRAIN_FILE,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Training curves

Total loss and its terms, per epoch. In `replay` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph computation

`replay` re-renders the paper's saved Morse graph. `morse` and `retrain` both
compute one at `SUBDIV`, so the grid is chosen here rather than inherited from
the training config.

Grid size is the binding constraint. CMGDB's cached map graph is bounded by
`CMGDB_MAPGRAPH_MAX_VERTICES`, `2**24` cells by default. A paper-resolution
grid can exceed that -- the two-dimensional contraction example needs `2**27` --
and the run then stops rather than silently falling back to a per-cell map
callback, which is orders of magnitude slower. Raising the limit costs roughly
8 bytes per cell plus 8 bytes per edge, so paper resolution wants a
large-memory machine. On a Colab runtime, keep `SUBDIV` small.

In [ ]:
if MODE == "replay":
    print("replay: using the paper's saved Morse graph")
else:
    exp = exp.recompute_morse(subdiv=SUBDIV, cmgdb_overrides=MORSE_OVERRIDES)
exp

## Morse graph

Two minimal nodes -- two stable equilibria.

In [ ]:
exp.show_morse_graph()

## Morse sets (1-D latent)

The latent line, with each Morse set drawn as a colored band.

In [ ]:
exp.show_morse_sets()

## Final-population histograms: base vs. adaptive design

Where trajectories end up after the transient, for the base design
$\mathcal{D}(500)$ and the adaptively-augmented design. (Replay artifacts only;
these read the preserved histogram data.)

In [ ]:
from latentdynamics.replay import repo_path, show_image
from latentdynamics.viz.histograms import (
    plot_final_population_histogram,
    steps_per_trajectory_from_metadata,
)

out = repo_path("notebooks/rendered/coral")
data_dir = repo_path("replay_sources/coral/histogram_data")
panels = [
    ("D(500, 0, 20)", "train_500"),
    ("D(500) + 300 adaptive samples", "train_500_300_adaptive"),
]
for title, name in panels:
    meta = data_dir / f"{name}_metadata.json"
    if not (data_dir / f"{name}.csv").exists():
        print(f"skip {title}: no histogram data at {data_dir}")
        continue
    steps = steps_per_trajectory_from_metadata(meta)
    png = plot_final_population_histogram(
        data_dir / f"{name}.csv", out / f"hist_{name}.png", steps_per_trajectory=steps
    )
    print(title)
    show_image(png, width=560)

## Run provenance

`metrics` reports the section 5.4.1 success check: `a0`/`a1` should be `true`
(each reference attractor sits in a unique minimal Morse set) and `r` should be
`false`.

In [ ]:
exp.diagnostics()